In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 21.58it/s]


Numba compilation complete!


In [2]:
#initial file processing
workcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = workcomp

#Comment depending on project type
#eopn3work = "N"
eopn3work = "Y"
eopn3work = "tgt"

if eopn3work == "Y":
    whomst ="NL"
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\2. Processed\\" + whomst + "\\"
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\3. Compiled\\"  + whomst + "\\"
    
if eopn3work == "tgt":
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\2. Processed\\" 
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\3. Compiled\\" 
    
else:
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
    savedir = titledpath + filedir + "Compilation with delta\\2025meandiffcollection\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"


In [3]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

# for file_no in os.listdir(openPath): 
#     if respondercsv in file_no and "w1118" not in file_no :   
#         f = os.path.join(openPath, file_no)
#         dfe=pd.read_csv(f)
#         exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
#         driver = file_no.split(" ")[0]
#         lstnew.append(driver)
# lst = lstnew.copy()

#processing ONLY specific names
lst = ["elav", "nSyb", "OK371", "Piezo", "vGAT", "Cha-6793"]

print(lst)

['elav', 'nSyb', 'OK371', 'Piezo', 'vGAT', 'Cha-6793']


In [4]:
diff = pd.DataFrame()
diffbs = pd.DataFrame()

for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    #mean_diff
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True)

    #delta_g    
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_t = NLMATH.timetype(dfwt, dfexpt).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_d = pd.concat([NLMATH.totaldisp(dfexpt, "Expt"), NLMATH.totaldisp(dfwt, "WT")]).reset_index(drop=True)
    df_bp = NLMATH.bheight(NLMATH.boutheight(dfexpt), NLMATH.boutheight(dfwt)).reset_index(drop=True)
    df_pp = NLMATH.bheight(NLMATH.pauseheight(dfexpt), NLMATH.pauseheight(dfwt)).reset_index(drop=True)
    df_dispp = pd.concat([NLMATH.displacementbetweenpauses(dfexpt, "Expt"), NLMATH.displacementbetweenpauses(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    #df_sim = pd.concat([NLMATH.straightnessindexmeter(dfexpt, "Expt"), NLMATH.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #ascentdescent
    updown_df = pd.DataFrame()
    updown_df = pd.concat([NLMATH.positional_arguments(dfexpt, driver), NLMATH.positional_arguments(dfwt, "w1118")], axis = 0)
    descendingdf = updown_df[updown_df['index'].str.contains("Descending.*")].reset_index(drop=True)
    ascendingdf = updown_df[updown_df['index'].str.contains("Ascending.*")].reset_index(drop=True)
    
    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLMATH.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLMATH.pausecomp(dfexpt, driver)
        
    alltgtmeandf_pause = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Pauses"), NLMATH.pausenumber(expttotalmeanevent, n, "Pauses")], axis = 0).reset_index(drop=True)
    alltgtmeandf_bout = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Bouts"), NLMATH.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    
    alltgtnumberdf_pause = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Pauses"), NLMATH.pausenumber(expttotalnumberevent, n, "Pauses")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Bouts"), NLMATH.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
            
    #___________________________________________#    
    # meandiff plots -- you run mean_diff instead of delta_g because since all the binary data is at the same dimension, no standardization is required and empirical delta delta is sufficient
    dff2_prop = NLMATH.deltaversion_binary(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLMATH.deltaversion_binary(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0

    #deltaG plots
    dfs2 = NLMATH.deltaversion(df_sp, "Velocity", "speed")
    dft2 = NLMATH.deltaversion(df_t, "Time", "time") #time spent above 3/4 of height
    dfh2 = NLMATH.deltaversion(df_h, "Y", "height")
    dfd2 = NLMATH.deltaversion(df_d, "displacement", "displacement")
    dfbs2 = NLMATH.deltaversion(df_bsp, "BSpeed", "bspeed")
    dfbp2 = NLMATH.deltaversion(df_bp, "Height", "boutpos")
    dfpp2 = NLMATH.deltaversion(df_pp, "Height", "pausepos")
    dfdbp2 = NLMATH.deltaversion(df_dispp, "avgdisplacementbetweenpause", "displacementbetweenpause")
    dfmv2 = NLMATH.deltaversion(df_maxv, "maxvelocity", "maxvelocity")
    #dfsim2 = NLMATH.deltaversion(df_sim, "averagestraightnessindex", "straightindex")
    dfasc = NLMATH.deltaversion(ascendingdf, "Position", "ascent")
    dfdesc = NLMATH.deltaversion(descendingdf, "Position", "descent")
    #pause and bouts
    dfmp2 = NLMATH.deltaversion(alltgtmeandf_pause, "Pauses", "meanpause")
    dfmb2 = NLMATH.deltaversion(alltgtmeandf_bout, "Bouts", "meanbout")     
    dfnp2 = NLMATH.deltaversion(alltgtnumberdf_pause, "Pauses", "pause")
    dfnb2 = NLMATH.deltaversion(alltgtnumberdf_bout, "Bouts", "bout")
    
    
    #dftotal = pd.concat([dff2_prop, dff2_number, dfs2, dft2, dfh2, dfd2, dfbs2, dfpp2, dfdbp2, dfmv2, dfsim2, dfasc, dfdesc, dfmp2, dfmb2, dfnp2, dfnb2], axis = 1)
    dftotal = pd.concat([dff2_prop, dff2_number, dfs2, dft2, dfh2, dfd2, dfbs2, dfpp2, dfdbp2, dfmv2, dfasc, dfdesc, dfmp2, dfmb2, dfnp2, dfnb2], axis = 1)
    dftotal['MBON'] = n


    dftotal.set_index("MBON", inplace = True)
    dftotal.to_csv(savedir + n + " x " + responder + " allstats.csv")

#2025 collection will have a mix of delta g and mean diff
#20241014 is all solely mean diffs
        


vGAT


ValueError: The divisor is zero, indicating no variability in the data.